# SAE Levels of Driving Automation and the Operational Design Domain

**Module 01 · Perception Systems** · Notebook 1 of 8

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/milinpatel07/Autonomous-Driving_AI-Safety-and-Security/blob/main/01_Perception/01_sae_levels_and_odd.ipynb)

| Level | Time | Prerequisites | Standards |
|---|---|---|---|
| Beginner | ~45 min | None | SAE J3016 (2021); ISO 34503:2023 |

**Author:** Milin Patel · Hochschule Kempten · MIT Licence

## Learning objectives

By the end of this notebook you will be able to:

1. Describe SAE J3016 and its six levels of driving automation (0–5) in terms of roles, not percentages
2. Distinguish driver support (L0–L2) from automated driving (L3–L5) and explain the DDT fallback
3. Classify real-world systems by SAE level, using current (2026) examples
4. Define the Operational Design Domain (ODD), its dimensions, and how a system can check it at runtime
5. Outline the sense → perceive → predict → plan → act architecture and where machine learning is used
6. Explain why ML-based perception creates new safety questions (ISO 21448, ISO/PAS 8800)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
from IPython.display import display

# Okabe-Ito colour-blind-safe palette
OKABE_ITO = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', '#D55E00', '#CC79A7', '#000000']
plt.rcParams.update({'figure.dpi': 100, 'axes.grid': True, 'grid.alpha': 0.3})
print(f"NumPy {np.__version__}, pandas {pd.__version__}")

---
## 1. SAE J3016: levels of driving automation

SAE J3016 (revision April 2021) defines **six levels** of driving automation. The levels are defined by **which roles** the human and the system perform, not by how "smart" the system is:

- **DDT (dynamic driving task):** sustained lateral *and/or* longitudinal control plus **OEDR** (object and event detection and response — i.e. monitoring the environment and reacting).
- **DDT fallback:** who responds when the system fails or reaches its ODD limit.
- **ODD:** the conditions under which the feature is designed to work.

J3016 itself states that the levels are *descriptive rather than normative and technical rather than legal*. Who is **legally liable** is decided by national law (e.g. the German Road Traffic Act, StVG §§ 1a–1l), not by J3016.

In [ ]:
sae_levels = pd.DataFrame({
    'Level': [0, 1, 2, 3, 4, 5],
    'Name': ['No Driving Automation', 'Driver Assistance', 'Partial Driving Automation',
             'Conditional Driving Automation', 'High Driving Automation', 'Full Driving Automation'],
    'System performs': [
        'No sustained control (warnings / momentary intervention, e.g. AEB)',
        'Sustained steering OR speed (accel/brake) control',
        'Sustained steering AND speed control',
        'Entire DDT while engaged (within ODD)',
        'Entire DDT and DDT fallback (within ODD)',
        'Entire DDT and DDT fallback (unlimited ODD)'],
    'Human role': [
        'Drives; supported by warnings',
        'Drives and supervises',
        'Supervises continuously (eyes on road; hands-on or hands-free depending on system)',
        'Fallback-ready user: must take over when requested',
        'Passenger while engaged; no takeover needed in ODD',
        'Passenger'],
    'Example (as of Oct 2026)': [
        'Lane-departure warning, AEB',
        'Adaptive cruise control (ACC)',
        'Tesla Autopilot / FSD (Supervised), GM Super Cruise, Ford BlueCruise',
        'Mercedes-Benz DRIVE PILOT (German Autobahn, up to 95 km/h)',
        'Waymo driverless ride-hailing (geofenced US metro areas)',
        'No such system exists'],
})
display(sae_levels)

print("--- SAE J3016 levels (text summary) ---")
for _, r in sae_levels.iterrows():
    print(f"  L{r['Level']} {r['Name']}: system = {r['System performs']}; human = {r['Human role']}; "
          f"example = {r['Example (as of Oct 2026)']}")

### Critical distinction: Level 2 vs Level 3

**Level 2 (partial driving automation)**
- The system steers and controls speed, but the **driver performs OEDR** — they must continuously monitor the road and intervene.
- Hands-free L2 systems (Super Cruise, BlueCruise) are still L2: "hands-off, eyes-on".
- Example: Tesla Autopilot / FSD (Supervised), Mercedes-Benz Active Steering Assist.

**Level 3 (conditional driving automation)**
- While engaged, the **system performs the entire DDT, including OEDR**. The user may divert attention but must be a *fallback-ready user* who responds to a takeover request within an adequate time.
- Example: Mercedes-Benz DRIVE PILOT — since the KBA approval of December 2024 usable on the German Autobahn at up to 95 km/h in the right-hand lane, in flowing traffic behind a lead vehicle (the 2022 version was limited to 60 km/h in congested traffic) ([Mercedes-Benz, Dec 2024](https://group.mercedes-benz.com/technology/autonomous-driving/driving/drive-pilot-95-kmh.html)).

**Key insight:** from L2 to L3 the *monitoring role (OEDR)* moves from the human to the system. That shift is what makes perception performance the core safety question for L3+. Who is liable after the shift is a matter of national law.

The bar charts often used to show "% responsibility" per level are not based on J3016 — the standard assigns **roles**, not percentages. The matrix below shows the J3016 roles directly.

In [ ]:
roles = pd.DataFrame({
    'Level': [0, 1, 2, 3, 4, 5],
    'Sustained lateral + longitudinal control': ['Driver', 'Driver + System (one axis)', 'System', 'System', 'System', 'System'],
    'OEDR (monitoring)': ['Driver', 'Driver', 'Driver', 'System', 'System', 'System'],
    'DDT fallback': ['Driver', 'Driver', 'Driver', 'Fallback-ready user', 'System', 'System'],
    'ODD': ['n/a', 'Limited', 'Limited', 'Limited', 'Limited', 'Unlimited'],
})
display(roles)

# Encode the matrix numerically: 0 = human, 0.5 = shared / fallback-ready user, 1 = system
code_map = {'Driver': 0.0, 'Driver + System (one axis)': 0.5, 'Fallback-ready user': 0.5, 'System': 1.0}
cols = ['Sustained lateral + longitudinal control', 'OEDR (monitoring)', 'DDT fallback']
M = np.array([[code_map[roles.loc[i, c]] for c in cols] for i in range(6)])

fig, ax = plt.subplots(figsize=(9, 4.5))
im = ax.imshow(M.T, cmap='cividis', vmin=0, vmax=1, aspect='auto')
ax.set_xticks(range(6)); ax.set_xticklabels([f'L{i}' for i in range(6)])
ax.set_yticks(range(3)); ax.set_yticklabels(['Sustained control', 'OEDR (monitoring)', 'DDT fallback'])
for i in range(6):
    for j, c in enumerate(cols):
        val = roles.loc[i, c]
        ax.text(i, j, val.replace(' (one axis)', '\n(one axis)').replace('Fallback-ready user', 'Fallback-\nready user'),
                ha='center', va='center', fontsize=8, color='white' if M[i, j] < 0.5 else 'black')
ax.set_xlabel('SAE level'); ax.set_title('Who performs each role? (SAE J3016:2021)')
ax.grid(False)
cb = fig.colorbar(im, ax=ax, ticks=[0, 0.5, 1]); cb.ax.set_yticklabels(['Human', 'Shared', 'System'])
plt.tight_layout(); plt.show()

first_sys_oedr = int(roles.index[roles['OEDR (monitoring)'] == 'System'][0])
print("--- Role matrix (text summary) ---")
for i in range(6):
    print(f"  L{i}: control={roles.loc[i, cols[0]]}, OEDR={roles.loc[i, cols[1]]}, fallback={roles.loc[i, cols[2]]}, ODD={roles.loc[i, 'ODD']}")
print(f"  Monitoring (OEDR) first moves to the system at Level {first_sys_oedr}.")

---
## 2. Operational Design Domain (ODD)

J3016 defines the **ODD** as the operating conditions under which a driving automation feature is *specifically designed to function*. ISO 34503:2023 and BSI PAS 1883:2020 provide taxonomies to specify an ODD. Typical dimensions:

- **Scenery:** road type, lane markings, junctions, geofence
- **Environmental conditions:** weather, illumination (day/night), visibility
- **Dynamic elements:** speed range, traffic density, presence of vulnerable road users

Examples (published information only; values that manufacturers do not publish are marked as such):

In [ ]:
odd_examples = pd.DataFrame({
    'System': ['Tesla FSD (Supervised)', 'Mercedes-Benz DRIVE PILOT', 'Waymo driverless ride-hailing'],
    'SAE level': [2, 3, 4],
    'Scenery / geography': [
        'No formal ODD published (driver supervises)',
        'German Autobahn; right-hand lane, flowing traffic, behind a lead vehicle',
        'Geofenced service areas in more than ten US metro areas (as of 2026)'],
    'Speed': [
        'Not formally published',
        'Up to 95 km/h (KBA approval, Dec 2024)',
        'Surface streets; freeway rides in some cities since Nov 2025'],
    'Environment': [
        'Not formally published',
        'Restrictions defined by the manufacturer (see owner\'s manual)',
        'Defined by the operator; not fully published'],
})
display(odd_examples)

print("--- ODD examples (text summary) ---")
for _, r in odd_examples.iterrows():
    print(f"  {r['System']} (L{r['SAE level']}): {r['Scenery / geography']}; speed: {r['Speed']}; environment: {r['Environment']}")

Sources (as of October 2026): Mercedes-Benz press release on DRIVE PILOT 95 km/h (Dec 2024); [TechCrunch, 12 Nov 2025](https://techcrunch.com/2025/11/12/waymo-robotaxis-are-now-giving-rides-on-freeways-in-these-3-cities/) (Waymo freeway rides in SF, LA and Phoenix); [WLRN/AP, Feb 2026](https://www.wlrn.org/science-technology/2026-02-25/waymos-robotaxis-now-being-dispatched-in-10-major-u-s-markets-with-expansion-in-texas-and-florida) (Waymo in 10 US markets; further cities launched later in 2026).

*Historical note:* earlier versions of this table listed the TuSimple L4 truck and Cruise robotaxi. TuSimple left autonomous trucking and rebranded as CreateAI in December 2024 ([TechCrunch](https://techcrunch.com/2024/12/19/tusimple-pivot-from-self-driving-to-ai-animation-is-complete-with-createai-rebrand/)); GM stopped funding Cruise's robotaxi development in December 2024 ([NBC News](https://www.nbcnews.com/business/autos/gm-cruise-robotaxi-business-rcna184294)).

### Making the ODD machine-checkable

An ODD is only useful for safety if the system can tell **at runtime** whether it is inside it. A minimal sketch: express each ODD dimension as an allowed set or range and report every violated dimension.

In [ ]:
# Illustrative ODD for an L3 motorway feature (values chosen for teaching, not a real product)
ODD = {
    'road_type':      {'motorway'},
    'speed_kmh':      (0, 95),
    'precipitation':  {'none', 'light_rain'},
    'illumination':   {'day', 'dusk', 'night_lit'},
    'lead_vehicle':   {True},
}

def odd_violations(state, odd=ODD):
    """Return the list of ODD dimensions violated by the current state."""
    out = []
    for key, allowed in odd.items():
        v = state[key]
        ok = (allowed[0] <= v <= allowed[1]) if isinstance(allowed, tuple) else (v in allowed)
        if not ok:
            out.append(f"{key}={v!r} not in {allowed}")
    return out

states = {
    'Nominal motorway, 80 km/h':        dict(road_type='motorway', speed_kmh=80, precipitation='none', illumination='day', lead_vehicle=True),
    'Heavy rain appears':               dict(road_type='motorway', speed_kmh=80, precipitation='heavy_rain', illumination='day', lead_vehicle=True),
    'Exit onto rural road, no lead car': dict(road_type='rural', speed_kmh=70, precipitation='none', illumination='night_unlit', lead_vehicle=False),
}
for name, s in states.items():
    v = odd_violations(s)
    verdict = 'inside ODD' if not v else f'OUTSIDE ODD -> issue takeover request ({len(v)} violation(s))'
    print(f"{name:35s}: {verdict}")
    for item in v:
        print(f"      - {item}")

Real ODD monitors must also handle **uncertain** estimates of the state (is it "light" or "heavy" rain?) and leave enough time for the takeover. This is developed in [07_Runtime_Assurance_and_Operations/01 — ODD and runtime monitoring](../07_Runtime_Assurance_and_Operations/01_odd_and_runtime_monitoring.ipynb).

### Why the ODD matters for safety

Operating outside the ODD — or inside an ODD whose boundaries were drawn too optimistically — creates hazards:

- **Use of a feature outside its intended conditions:** for example, an L2 lane-keeping feature engaged on roads without clear lane markings, where its perception was never designed to work.
- **Perception insufficiency *inside* the test ODD — Uber ATG, Tempe (2018):** the pedestrian was first detected about 5.6 s before impact but was repeatedly reclassified (vehicle / bicycle / other), so no path was predicted; the design did not consider jaywalking pedestrians, the factory AEB was disabled and the safety operator was distracted ([NTSB HAR-19/03](https://www.ntsb.gov/investigations/AccidentReports/Reports/HAR1903.pdf)). This is a lesson in perception insufficiency and safety culture, not an ODD exit.

**Mitigation example:** Waymo restricts service to mapped geofences (ODD enforcement).

**Safety principle:** the system must either (1) **detect ODD boundaries** in time and hand over or reach a minimal-risk condition, or (2) **be robust to everything it may encounter** (the Level 5 aspiration).

---
## 3. Automated driving system architecture

Most automated driving stacks follow a **sense → perceive → predict → plan → act** pipeline (end-to-end learned stacks merge some of these blocks; see notebook [04](04_object_detection.ipynb)).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0.2, 10.4); ax.set_ylim(0.8, 6); ax.axis('off')

modules = [
    {'name': 'SENSORS\n\nCamera\nLiDAR\nRadar\nGNSS/IMU', 'x': 0.5, 'ml': 'none'},
    {'name': 'PERCEPTION\n\nDetection\nClassification\nTracking\nLocalisation', 'x': 2.5, 'ml': 'high'},
    {'name': 'PREDICTION\n\nTrajectory\nforecast\nBehaviour\nmodelling', 'x': 4.5, 'ml': 'high'},
    {'name': 'PLANNING\n\nRoute\nBehaviour\nMotion\nplanning', 'x': 6.5, 'ml': 'moderate'},
    {'name': 'CONTROL\n\nSteering\nBraking\nAcceleration', 'x': 8.5, 'ml': 'low'},
]
ml_colour = {'none': '#BBBBBB', 'low': OKABE_ITO[1], 'moderate': OKABE_ITO[0], 'high': OKABE_ITO[5]}
for m in modules:
    ax.add_patch(FancyBboxPatch((m['x'], 2), 1.5, 2.5, boxstyle="round,pad=0.1",
                                facecolor=ml_colour[m['ml']], edgecolor='black', linewidth=1.5, alpha=0.8))
    ax.text(m['x'] + 0.75, 3.25, m['name'], ha='center', va='center', fontsize=9, fontweight='bold')
    ax.text(m['x'] + 0.75, 4.8, f"ML use: {m['ml']}", ha='center', fontsize=9)
for a, b in zip(modules[:-1], modules[1:]):
    ax.add_patch(FancyArrowPatch((a['x'] + 1.6, 3.25), (b['x'] - 0.1, 3.25),
                                 arrowstyle='->', mutation_scale=25, linewidth=2, color='black'))
ax.text(5, 5.5, 'Modular automated driving system architecture', ha='center', fontsize=14, fontweight='bold')
ax.text(5, 1.2, 'Data flows left to right: sensors -> perception -> prediction -> planning -> control',
        ha='center', fontsize=10, style='italic')
plt.tight_layout(); plt.show()

print("--- Architecture (text summary) ---")
for m in modules:
    print(f"  {m['name'].split(chr(10))[0]}: typical ML use = {m['ml']}")
heavy = [m['name'].split(chr(10))[0] for m in modules if m['ml'] == 'high']
print(f"  Modules with the heaviest ML use: {', '.join(heavy)}")

| Module | Function | Typical ML use | Example tasks |
|--------|----------|----------|---------------|
| **Sensors** | Collect raw data | None (signal processing) | Capture images, scan LiDAR, measure radar |
| **Perception** | Interpret the environment | High | Detect cars, pedestrians, lanes, signs |
| **Prediction** | Forecast future states | High | Predict pedestrian trajectories, vehicle intentions |
| **Planning** | Decide vehicle actions | Moderate (growing) | Route, behaviour and motion planning |
| **Control** | Execute commands | Low | PID / MPC controllers, actuator commands |

**This module's focus:** perception — where ML plays the largest role.

---
## 4. Why machine learning for perception?

### The challenge
- Open-ended variety of real-world scenes; complex, unstructured environments
- Real-time processing (typically 10–30 Hz sensor rates, e.g. nuScenes: 20 Hz LiDAR, 12 Hz cameras) within an end-to-end latency budget of the order of 100 ms
- Must handle uncertainty and ambiguity

### Rule-based approach
```python
if (object_height > object_width) and object_moving:
    label = "pedestrian"
elif object_rectangular and object_has_wheels:
    label = "car"
```
Fails with unusual poses, occlusion and lighting; requires manual feature engineering; cannot anticipate all cases.

### Machine-learning approach
```python
model.fit(images, labels)          # learn from millions of labelled examples
prediction = model.predict(new_image)
```
Learns from data and handles natural variation **within the training distribution**.

**Trade-off:** less predictable than explicit rules; can fail in unexpected ways under distribution shift ([Module 02 — Failures and Robustness](../02_Failures_and_Robustness/README.md)); requires new safety arguments ([Module 03 — ISO 26262](../03_Functional_Safety_ISO26262/README.md), [Module 04 — SOTIF](../04_SOTIF_ISO21448/README.md), [Module 06 — ISO/PAS 8800](../06_AI_Safety_ISO_PAS_8800/README.md)).

In [ ]:
comparison = pd.DataFrame({
    'Aspect': ['Development', 'Typical cases', 'Rare / edge cases', 'Explainability',
               'Safety validation', 'Improvement over time', 'Computational cost'],
    'Rule-based': ['Manual feature engineering', 'Good if rules cover the case', 'Poor (cannot anticipate all)',
                   'High (explicit rules)', 'Easier (test each rule)', 'Manual updates', 'Low'],
    'Machine learning': ['Large labelled dataset required', 'Excellent',
                         'Better within training distribution; unpredictable under distribution shift',
                         'Low ("black box")', 'Difficult (huge scenario space)',
                         'Requires new data, retraining and re-validation', 'High (accelerators needed)'],
})
display(comparison)
print("--- Rule-based vs ML (text summary) ---")
for _, r in comparison.iterrows():
    print(f"  {r['Aspect']}: rule-based = {r['Rule-based']}; ML = {r['Machine learning']}")

---
## 5. Safety perspective

The SAE level tells you **who** must detect objects and events; it does not tell you **how well** the system must do it. Three standards divide that question:

| Question | Standard | Perception example |
|---|---|---|
| What if hardware/software **malfunctions** (E/E faults)? | ISO 26262:2018 | Camera image sensor fails; bit flip in the detector's memory |
| What if the function works as designed but its **performance is insufficient** in some conditions (*triggering conditions*)? | ISO 21448:2022 (SOTIF) | Detector misses a pedestrian at night in rain |
| What if the insufficiency stems from the **ML model or its data**? | ISO/PAS 8800:2024 | Training data contained few wheelchair users |

The ODD is the starting point of the SOTIF analysis: triggering conditions are searched *within* the ODD, and leaving the ODD must itself be detected. Continue with [04_SOTIF_ISO21448/01 — SOTIF fundamentals](../04_SOTIF_ISO21448/01_sotif_fundamentals.ipynb) and [02 — Triggering conditions](../04_SOTIF_ISO21448/02_triggering_conditions_and_scenarios.ipynb).

---
## Exercise 1: SAE level classification

Classify the following systems by SAE level.

1. **Tesla Model 3 with Autopilot** (motorway, driver monitored by camera)
2. **Waymo driverless ride-hailing** (Phoenix, AZ, no one in the driver's seat)
3. **Mercedes-Benz S-Class with DRIVE PILOT** (German Autobahn, up to 95 km/h behind a lead vehicle)
4. **Car with adaptive cruise control only** (maintains speed and distance)
5. **Cruise robotaxi, San Francisco, 2023** (no driver, geofenced; service ended Dec 2024 — historical)
6. **Car with lane-centring assist only** (steers to keep the lane, no speed control)

In [ ]:
# TODO: fill in your answers (0-5)
your_answers = {
    'Tesla Autopilot': None,
    'Waymo Phoenix': None,
    'Mercedes DRIVE PILOT': None,
    'ACC only': None,
    'Cruise SF (2023)': None,
    'Lane centring only': None,
}

correct_answers = {'Tesla Autopilot': 2, 'Waymo Phoenix': 4, 'Mercedes DRIVE PILOT': 3,
                   'ACC only': 1, 'Cruise SF (2023)': 4, 'Lane centring only': 1}

if all(v is None for v in your_answers.values()):
    print("Fill in your answers above, then re-run this cell to check them.")
else:
    for system, level in your_answers.items():
        mark = 'correct' if level == correct_answers[system] else f'incorrect (answer: L{correct_answers[system]})'
        print(f"  {system}: you said L{level} -> {mark}")

## Exercise 2: ODD analysis

**Scenario:** you are designing a Level 4 shuttle for a university campus. Define an appropriate ODD using the structure of `ODD` above, then test it with `odd_violations`.

In [ ]:
# TODO: define your ODD
campus_shuttle_odd = {
    'Geography': '',          # e.g. "Campus roads, speed bumps, pedestrian zones"
    'Weather': '',            # e.g. "Clear, light rain"
    'Speed': '',              # e.g. "Max 25 km/h"
    'Time': '',               # e.g. "Daylight hours only"
    'Special_Conditions': '', # e.g. "No operation during large events"
}
print("Your ODD for the campus shuttle:")
for k, v in campus_shuttle_odd.items():
    print(f"  {k}: {v if v else '(not yet defined)'}")
print("\nReflection: why these boundaries? Which scenarios outside them are dangerous?"
      " How will the shuttle detect that it is leaving the ODD?")

## Key takeaways

- SAE J3016 levels are defined by **roles** (sustained control, OEDR, DDT fallback, ODD), not by percentages of responsibility; J3016 is technical, not legal.
- From L2 to L3 the **monitoring task (OEDR)** moves from the human to the system — perception performance becomes the central safety question.
- An **ODD** defines where, when and under which conditions a feature is designed to work; it must be specified (ISO 34503:2023) and monitored at runtime.
- Industry examples change quickly: DRIVE PILOT reached 95 km/h (Dec 2024), Waymo operates in more than ten US metro areas, Cruise and TuSimple have exited (all as of Oct 2026).
- Perception and prediction are the most ML-heavy blocks; ML generalises within its training distribution but is unpredictable under distribution shift.
- Malfunctions belong to ISO 26262, performance insufficiencies and triggering conditions to ISO 21448, ML-specific insufficiencies to ISO/PAS 8800.

## Self-assessment

1. What is the key difference between SAE Level 2 and Level 3 in terms of who monitors the environment (OEDR) and who performs the DDT fallback?
2. A shuttle operates only on mapped campus routes, in daylight, below 25 km/h, with no safety driver. What SAE level is this, and what defines its ODD?
3. Why is the perception module considered the most ML-dependent part of the pipeline?
4. Give one advantage and one disadvantage of ML instead of rule-based object detection.
5. An automated vehicle encounters heavy fog that is outside its ODD. What should an L3 system do, and what should an L4 system do?
6. Why is the Uber ATG crash (2018) better described as a perception insufficiency than as an ODD exit?

<details>
<summary>Show answers</summary>

1. At L2 the driver performs OEDR and the fallback; at L3 the system performs the whole DDT including OEDR while engaged, and the fallback-ready user must respond to takeover requests.
2. Level 4 (no human fallback needed within the ODD). The ODD is defined by geography (mapped campus routes), illumination (daylight), speed (< 25 km/h) and any further weather/traffic limits.
3. Turning raw pixels and points into objects, classes and positions is a pattern-recognition task without a closed-form rule set; deep learning is the only approach that currently scales to it.
4. Advantage: handles natural variation within the training distribution. Disadvantage: behaviour under distribution shift is hard to predict, explain and validate.
5. L3: issue a takeover request in time and, if the user does not respond, perform a minimal-risk manoeuvre where the system design provides one. L4: reach a minimal-risk condition itself (e.g. slow down and stop safely) — no human takeover is assumed.
6. The crash happened inside the test ODD. The pedestrian was detected ~5.6 s before impact but repeatedly reclassified, so no trajectory was predicted; disabled AEB and a distracted operator removed the safety net (NTSB HAR-19/03).

</details>

## References

1. SAE International (2021). *J3016_202104: Taxonomy and Definitions for Terms Related to Driving Automation Systems for On-Road Motor Vehicles.* https://www.sae.org/standards/content/j3016_202104/
2. ISO 34503:2023. *Road vehicles — Test scenarios for automated driving systems — Specification for operational design domain.*
3. BSI PAS 1883:2020. *Operational Design Domain (ODD) taxonomy for an automated driving system (ADS) — Specification.*
4. ISO 21448:2022. *Road vehicles — Safety of the intended functionality.* https://www.iso.org/standard/77490.html
5. ISO/PAS 8800:2024. *Road vehicles — Safety and artificial intelligence.*
6. NTSB (2019). *Collision Between Vehicle Controlled by Developmental Automated Driving System and Pedestrian, Tempe, Arizona, March 18, 2018.* HAR-19/03. https://www.ntsb.gov/investigations/AccidentReports/Reports/HAR1903.pdf
7. Mercedes-Benz Group (Dec 2024). *DRIVE PILOT 95 km/h.* https://group.mercedes-benz.com/technology/autonomous-driving/driving/drive-pilot-95-kmh.html
8. Caesar, H. et al. (2020). nuScenes: A multimodal dataset for autonomous driving. CVPR. arXiv:1903.11027
9. Patel, M., Jung, R., Cakir, Y. (2023). Simulation-Based Testing of Foreseeable Misuse by the Driver Applicable for Highly Automated Driving. LNNS (Springer). https://doi.org/10.1007/978-981-99-3043-2_15
10. Patel, M., Jung, R. (2023). Simulation-Based Application of Safety of the Intended Functionality to Mitigate Foreseeable Misuse in Automated Driving Systems. SAE. https://doi.org/10.47953/sae-pp-00371

---
**Navigation:** ← [Module 01 index](README.md) · [Module index](README.md) · [02 Sensor modalities](02_sensor_modalities.ipynb) →